In [4]:
import os, random, math, pickle
import pandas as pd
import numpy as np
from datetime import datetime

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from torch.utils.data import random_split
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

# Set environment variables for reproducibility and safety
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import precision_score, recall_score, accuracy_score

# 1. Configuration & Seeding
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
name = 'book'

## 1. Learn Embedding

### 1.1 Dataset

In [ ]:
class TCKGDataset(Dataset):
    def __init__(self, triplets):
        self.triplets = triplets
    def __len__(self):
        return len(self.triplets)
    def __getitem__(self, idx):
        # Trảmovie về bộ ba (head, relation, tail)
        return self.triplets[idx]

#### 1.2 TransE Model

In [ ]:
class TransE(pl.LightningModule):
    def __init__(self, num_entities, num_relations, embedding_dim=64, lr=1e-3, weight_decay=1e-4, dropout_rate=0.2):
        super().__init__()
        self.save_hyperparameters()
        
        # Khởi tạo Embeddings
        self.entity_emb = nn.Embedding(num_entities + 1, embedding_dim, padding_idx=0)     # +1 because starting at 1 instead of 0
        self.relation_emb = nn.Embedding(num_relations + 1, embedding_dim, padding_idx=0)
        
        # # Xavier initialization giúp hội tụ tốt hơn
        # nn.init.xavier_uniform_(self.entity_emb.weight)
        # nn.init.xavier_uniform_(self.relation_emb.weight)

        self.dropout = nn.Dropout(dropout_rate)
        
    def forward(self, h, r, t):
        h_e = self.entity_emb(h)
        r_e = self.relation_emb(r)
        t_e = self.entity_emb(t)

        # 2. Embedding Normalization (Rất quan trọng cho TransE)
        # Ép độ dài các vector về 1 (Unit Norm constraint)
        h_e = F.normalize(h_e, p=2, dim=1)
        r_e = F.normalize(r_e, p=2, dim=1)
        t_e = F.normalize(t_e, p=2, dim=1)
        
        # 3. Áp dụng Dropout
        h_e = self.dropout(h_e)
        r_e = self.dropout(r_e)
        t_e = self.dropout(t_e)
        
        # Công thức (6): Khoảng cách bình phương L2
        # g_r(h, t) = ||h + r - t||^2
        score = torch.sum((h_e + r_e - t_e)**2, dim=1)
        return score

    def training_step(self, batch, batch_idx):
        h, r, t = batch[:, 0], batch[:, 1], batch[:, 2]
        
        # Tính score cho bộ ba đúng (Positive) -> Cần giảm thiểu khoảng cách này
        pos_scores = self(h, r, t)
        
        # Negative Sampling: Thay thế tail t bằng t' ngẫu nhiên
        # t' không nhất thiết phải là không đúng thực tế (simplified), nhưng xác suất cao là không đúng.
        rand_t = torch.randint(1, self.hparams.num_entities + 1, t.shape, device=self.device)
        
        # Tính score cho bộ ba sai (Negative) -> Cần tối đa hóa khoảng cách này
        neg_scores = self(h, r, rand_t)
        
        # Công thức (7) Loss: -ln(sigmoid(g_neg - g_pos))
        # Chúng ta muốn g_neg > g_pos (khoảng cách sai lớn hơn đúng)
        # => (g_neg - g_pos) càng lớn càng tốt
        loss = -F.logsigmoid(neg_scores - pos_scores).mean()
        
        # Log loss
        self.log('train_loss', loss, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        h, r, t = batch[:, 0], batch[:, 1], batch[:, 2] #h,r,t shape = batch_size
        
        # 1. Tính loss trên valid set
        pos_scores = self(h, r, t)
        
        # Negative sampling (đơn giản hoá để tính loss theo dõi)
        rand_t = torch.randint(1, self.hparams.num_entities + 1, t.shape, device=self.device)

        neg_scores = self(h, r, rand_t)
        
        val_loss = -F.logsigmoid(neg_scores - pos_scores).mean()
        self.log('val_loss', val_loss, prog_bar=True)
        return val_loss

    def configure_optimizers(self):
        # 4. Thêm weight_decay (L2 regularization) vào Adam
        return torch.optim.Adam(self.parameters(), lr=self.hparams.lr,
                                weight_decay=self.hparams.weight_decay)

#### 1.2 KGCN Model

In [5]:
from collections import defaultdict

class KnownledgeGraph():
    def __init__(self, graph_file, nbr_sample_size=1):
        self.graph_file = graph_file
        self.graph_nbr_sample_size = nbr_sample_size
        self.graph_entities = None
        self.graph_relations = None
        self.graph_n_entity = 0
        self.graph_n_relation = 0
        self.graph = defaultdict(set)

    def build(self):
        df = pd.read_csv(self.graph_file)

        # head_id, relation_id, tail_id, head_id:token, relation_id:token, tail_id:token
        head_ids = df['head_id'].values
        tail_ids = df['tail_id'].values

        self.graph_entities = np.unique(np.concatenate([head_ids, tail_ids]))
        self.graph_n_entity = len(self.graph_entities)

        self.graph_relations = np.unique(df['relation_id'].values)
        self.graph_n_relation = len(self.graph_relations)

        # Build the knowledge graph with sets
        for head_id, relation_id, tail_id, _, _, _ in df.values:
            self.graph[head_id].add((tail_id, relation_id))
            self.graph[tail_id].add((head_id, relation_id))

    def get_neighbors(self, entity_id, sample_size=8):
        """Get neighbors for an entity with optional sampling"""
        if sample_size is None:
            sample_size = self.graph_nbr_sample_size

        entity_id = entity_id.item() if torch.is_tensor(entity_id) else entity_id
        neighbors = list(self.graph.get(entity_id, set()))
        n_neighbors = len(neighbors)

        if n_neighbors == 0:
            return torch.tensor([], dtype=torch.long), torch.tensor([], dtype=torch.long)

        if n_neighbors >= sample_size:
            sampled_indices = np.random.choice(n_neighbors, size=sample_size, replace=False)
        else:
            sampled_indices = np.random.choice(n_neighbors, size=sample_size, replace=True)

        sampled_neighbors = [neighbors[i] for i in sampled_indices]
        adj_entities = torch.tensor([n for n, _ in sampled_neighbors], dtype=torch.long)
        adj_relations = torch.tensor([r for _, r in sampled_neighbors], dtype=torch.long)

        return adj_entities, adj_relations

In [6]:
class SumAggregator(nn.Module):
    def __init__(self, embedding_dim):
        super(SumAggregator, self).__init__()
        self.linear = nn.Linear(embedding_dim, embedding_dim)

    def forward(self, neighbor_embs, central_embs):
        """
        neighbor_embs: Tensor of shape (batch, emb_dim)
        central_embs: Tensor of shape (batch, emb_dim)
        """

        # Combine with central entity embedding
        combined = neighbor_embs + central_embs  # shape: (batch, emb_dim) ([1204, 64])

        # Linear + activation
        output = torch.tanh(self.linear(combined))  # shape: (batch, emb_dim) torch.Size([1204, 64])
        return output

In [7]:
class KGCN(pl.LightningModule):
    def __init__(self, num_users, num_items, graph: KnownledgeGraph,
                 embedding_dim, lr):
        super().__init__()
        self.save_hyperparameters()

        model_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model_device = model_device

        self.graph = graph
        self.edge_index = None

        self.num_users = num_users
        self.num_items = num_items

        self.embedding_dim = embedding_dim

        # Embedding layers
        self.user_embedding = nn.Embedding(num_users, embedding_dim).to(model_device)
        self.entity_embedding = nn.Embedding(graph.graph_n_entity, embedding_dim).to(model_device)
        self.relation_embedding = nn.Embedding(graph.graph_n_relation, embedding_dim).to(model_device)

        # Aggregator
        self.aggregator = SumAggregator(embedding_dim).to(model_device)

        self.dropout = nn.Dropout(p=0.1)

    def setup(self, stage=None):
        self.train_user_pos_items = self.trainer.datamodule.train_user_pos_items
        self.val_user_pos_items = self.trainer.datamodule.val_user_pos_items

    @staticmethod
    def hit_at_k(pred_items, true_items, k):
        hits = 0
        for pred, true in zip(pred_items, true_items):
            # Count if any true item is in top-k predictions
            if len(set(pred[:k]) & set(true)) > 0:
                hits += 1
        return hits / len(true_items)

    @staticmethod
    def ndcg_at_k(pred_items, true_items, k):
        ndcg = 0.0
        for pred, true in zip(pred_items, true_items):
            gains = []
            for idx, item in enumerate(pred[:k]):
                gains.append(1 if item in true else 0)
            ideal_gains = [1] * min(len(true), k)
            dcg = sum(g / math.log2(i+2) for i, g in enumerate(gains))
            idcg = sum(g / math.log2(i+2) for i, g in enumerate(ideal_gains))
            ndcg += dcg / idcg if idcg > 0 else 0
        return ndcg / len(true_items)

    @staticmethod
    def recall_at_k(pred_items, true_items, k):
        recall = 0.0
        for pred, true in zip(pred_items, true_items):
            recall += len(set(pred[:k]) & set(true)) / len(true)
        return recall / len(true_items)

    @staticmethod
    def precision_at_k(pred_items, true_items, k):
        precision = 0.0
        for pred, true in zip(pred_items, true_items):
            precision += len(set(pred[:k]) & set(true)) / k
        return precision / len(true_items)

    def forward(self, batch):
        user_ids, item_ids, neighbor_ids, relation_ids = batch

        full_user_embs = self.user_embedding.weight
        full_entity_embs = self.entity_embedding.weight
        full_relation_embs = self.relation_embedding.weight

        user_embs = full_user_embs[user_ids] #torch.Size([1204, 64])
        neighbor_embs = full_entity_embs[neighbor_ids] # torch.Size([1204, 8, 64])
        relation_embs = full_relation_embs[relation_ids] #torch.Size([1204, 8, 64])

        item_embs = full_entity_embs[item_ids] #torch.Size([1204, 64])

        ################ User - Relation Attention ################
        # Expand user_embs to match neighbor dimension
        user_embs_expanded = user_embs.unsqueeze(1)  # [1204, 1, 64]

        # Compute scores: dot product between relation_embs and user_embs
        scores = torch.exp(-torch.abs(user_embs_expanded - relation_embs).sum(dim=2))  # [batch_size, k]

        attention_weights = F.softmax(scores, dim=1).unsqueeze(-1)  # [1204, 8, 1]
        weighted_neighbor_embs = neighbor_embs * attention_weights  # [1204, 8, 64]
        weighted_neighbor_embs = weighted_neighbor_embs.mean(dim=1) # [1204, 64]
        ################ User - Relation Attention ################

        updated_item_embs  = self.aggregator(weighted_neighbor_embs, item_embs) #([1204, 64])

        agg_full_entity_embs = full_entity_embs.clone()
        agg_full_entity_embs.index_copy_(0, item_ids, updated_item_embs)

        agg_full_entity_embs = self.dropout(agg_full_entity_embs)


        return full_user_embs, agg_full_entity_embs

    def compute_loss(self, batch, full_user_embs, agg_full_entity_embs):
        user_ids, item_ids, neighbor_ids, relation_ids = batch

        user_embs = full_user_embs[user_ids]
        pos_item_embs = agg_full_entity_embs[item_ids]

        pos_scores = torch.exp(-torch.abs(user_embs - pos_item_embs).sum(dim=1))

        ######################################### Hard negative Sampling #########################################
        full_item_embs = agg_full_entity_embs[:self.num_items]

        distances = torch.cdist(user_embs, full_item_embs, p=1) # [batch_size, num_items] torch.Size([1204, 4779])
        scores = torch.exp(-distances)  # [batch_size, num_items] torch.Size([1204, 4779])


        ################## Mask all pos_item_ids of the user in training_set ##################
        ### Basically, the  model should only see the information in the train_dataset.
        ### Therefore, only mask the pos_item_ids of the user in train_dataset
        ### All cell (user, item) in val_dataset should be treated as blank hence don't mask the val_dataset
        for i, u in enumerate(user_ids.tolist()):
            pos_item_ids = [item for item in self.train_user_pos_items[u]]
            scores[i, pos_item_ids] = float('-inf')
        ################## Mask all pos_item_ids of the user in training_set ##################

        k = 10 # Select top-K most negatives for each user
        neg_item_ids = torch.topk(scores, k=k, dim=1).indices  # [batch_size, k]

        # Get embeddings for these negatives
        neg_emb = agg_full_entity_embs[neg_item_ids]  # shape: [batch_size, k, emb_dim]

        neg_scores = torch.exp(-torch.abs(user_embs.unsqueeze(1) - neg_emb).sum(dim=2))  # [batch_size, k]
        neg_scores = neg_scores.mean(dim=1) # [batch_size]
        ######################################### Hard negative Sampling #########################################

        ####################### Compute Loss #######################
        scores = torch.cat([pos_scores, neg_scores], dim=0)
        labels = torch.cat([torch.ones_like(pos_scores), torch.zeros_like(neg_scores)], dim=0)

        loss = F.binary_cross_entropy(scores, labels)
        ####################### Compute Loss #######################

        return loss, pos_scores, neg_scores

    def training_step(self, batch, batch_idx):
        full_user_embs, agg_full_entity_embs = self(batch)
        loss, _, _ = self.compute_loss(batch, full_user_embs, agg_full_entity_embs)

        self.log("train_loss", loss, on_epoch=True, prog_bar=True)

        return loss

    def validation_step(self, batch, batch_idx):
        user_ids, item_ids, neighbor_ids, relation_ids = batch

        full_user_embs, agg_full_entity_embs = self(batch)
        full_item_embs = agg_full_entity_embs[:self.num_items]

        user_embs = full_user_embs[user_ids]

        distances = torch.cdist(user_embs, full_item_embs, p=1)  # [batch_size, num_items] torch.Size([1204, 4779])
        scores = torch.exp(-distances)  # [batch_size, num_items] torch.Size([1204, 4779]), it is the score between the ith user in batch_size and ALL items

        # ########## Mask those user-item pair that already in training set so that it won't suggest again
        mask = torch.zeros_like(scores, dtype=torch.bool)
        for i, u in enumerate(user_ids.tolist()):
            trained_items = [item for item in self.train_user_pos_items[u]]
            mask[i, trained_items] = True

        scores = scores.masked_fill(mask, float('-inf'))    #### Make them to -inf so that TopK won't pick again
        ########## Mask those user-item pair that already in training set so that it won't suggest again


        ################ Calculate metrics
        k_values = [10]  # Example: you can add more values as needed

        for k in k_values:
            # Get top-k items for this k
            topk_items = torch.topk(scores, k=k, dim=1).indices.tolist() # (1024, K=5)

            true_items = []  # (1024, variable length), each user may have multiple positive items
            for u in user_ids.tolist():
                adjusted_val_items = [item for item in self.val_user_pos_items[u]]
                true_items.append(adjusted_val_items)

            # Compute metrics for this k
            hit = self.hit_at_k(topk_items, true_items, k)
            ndcg = self.ndcg_at_k(topk_items, true_items, k)
            recall = self.recall_at_k(topk_items, true_items, k)
            precision = self.precision_at_k(topk_items, true_items, k)

            # Log metrics dynamically
            self.log(f"val_hit@{k:02d}", hit, prog_bar=True)
            self.log(f"val_recall@{k:02d}", recall, prog_bar=True)
            self.log(f"val_precision@{k:02d}", precision, prog_bar=True)
            self.log(f"val_ndcg@{k:02d}", ndcg, prog_bar=True)


        # Compute bpr_loss
        loss,  pos_scores, neg_scores = self.compute_loss(batch, full_user_embs, agg_full_entity_embs)

        self.log('val_loss', loss, prog_bar=True, logger=True)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.hparams.lr, weight_decay=1e-5)
        scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.75)
        return [optimizer], [scheduler]

#### 1.3 DataModule

In [ ]:
class RecommenderDataModule(pl.LightningDataModule):
    def __init__(self, interaction_file, graph: KnownledgeGraph, batch_size, val_size):
        super().__init__()
        self.interaction_file = interaction_file
        self.batch_size = batch_size
        self.val_size = val_size
        self.graph = graph

    def prepare_data(self):
        # Load interactions
        df_inter = pd.read_csv(self.interaction_file)

        unique_users = df_inter['user_id'].unique()
        unique_items = df_inter['entity_id'].unique()

        self.num_users = len(unique_users)
        self.num_items = len(unique_items)

        # Build positive interactions
        dataset = []    # user_id, entity_id (item_id), neighbor_ids, relation_ids,
        for _, row in df_inter.iterrows():
            u = row['user_id']
            e = row['entity_id']
            n, r = self.graph.get_neighbors(e)
            dataset.append((u, e, n, r))


        # Split interactions for validation
        train_size = int(len(dataset) * (1 - self.val_size))
        val_size = len(dataset) - train_size
        self.train_dataset, self.val_dataset = random_split(dataset, [train_size, val_size])

        train_user_pos_items = defaultdict(set)
        for u, i, _, _ in self.train_dataset:
            train_user_pos_items[u].add(i)

        val_user_pos_items = defaultdict(set)
        for u, i, _, _ in self.val_dataset:
            val_user_pos_items[u].add(i)

        self.train_user_pos_items = train_user_pos_items
        self.val_user_pos_items = val_user_pos_items

    def train_dataloader(self):
        return DataLoader(self.train_dataset, batch_size=self.batch_size, shuffle=True)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, shuffle=False)

### 1.3 Load dataset

In [ ]:
file_path = f'./data/{name}_TCKG.csv' 
print(f"Loading data from {file_path}...")

TCKG_df = pd.read_csv(file_path)

# Chuyển đổi dữ liệu sang index
triplets_np = np.stack([
    TCKG_df['head_id'],
    TCKG_df['relation_id'],
    TCKG_df['tail_id']
], axis=1)

# 2. Tìm Offset (Lấy ID lớn nhất của relation hiện tại)
# Ví dụ: nếu relation_id chạy từ 1 đến 10, offset sẽ là 10.
offset = TCKG_df['relation_id'].max()

# 3. Tạo Inverse Connections (Cạnh ngược)
# Đảo vị trí Tail -> Head, Head -> Tail, và cộng offset vào Relation
inverse_triplets_np = np.stack([
    TCKG_df['tail_id'],                 # Tail thành Head
    TCKG_df['relation_id'] + offset,    # Relation mới = Relation cũ + offset
    TCKG_df['head_id']                  # Head thành Tail
], axis=1)

# 4. Gộp cả 2 mảng lại với nhau
# axis=0 nghĩa là nối tiếp theo chiều dọc (thêm dòng)
all_triplets_np = np.concatenate([triplets_np, inverse_triplets_np], axis=0)

# Chuyển sang Tensor
triplets_tensor = torch.tensor(all_triplets_np, dtype=torch.long)
print(f'triplets_tensor.shape: {triplets_tensor.shape}')

# Tạo DataLoader
full_dataset = TCKGDataset(triplets_tensor)

# Chia 90% Train - 10% Val
train_size = int(0.9 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_set, val_set = random_split(full_dataset, [train_size, val_size])

# Tạo 2 Loaders
train_loader = DataLoader(train_set, batch_size=1024, shuffle=True, num_workers=0)
val_loader = DataLoader(val_set, batch_size=1024, shuffle=False, num_workers=0)


### 1.3 Init and train model

In [ ]:
timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

num_entites = pd.concat([TCKG_df['head_id'], TCKG_df['tail_id']]).max()

num_relations = TCKG_df['relation_id'].max() * 2    #*2 to double relation for inverse connection

print(f"Total Entities: {num_entites}")
print(f"Total Relations: {num_relations}")

checkpoint_callback = ModelCheckpoint(
    monitor='val_loss',       # Theo dõi val_loss
    dirpath=f'./checkpoints/', # Thư mục lưu
    filename=f'{name}-transE-{timestamp}-{{epoch:02d}}-{{val_loss:.4f}}', 
    save_top_k=1,             # Chỉ giữ lại 1 model tốt nhất
    mode='min',               # Lưu khi val_loss nhỏ nhất
)

# 5. Early Stopping Callback
early_stop_callback = EarlyStopping(
    monitor='val_loss', # Theo dõi val_loss
    min_delta=0.001,    # Cải thiện tối thiểu cần thiết
    patience=20,         # Chờ 5 epochs nếu không cải thiện thì dừng
    verbose=True,
    mode='min'
)

model = TransE(
    num_entities=num_entites, 
    num_relations=num_relations, 
    embedding_dim=64, # Có thể chỉnh d-dimension tại đây
    lr=0.001,
    weight_decay=1e-3,  # Tăng lên nếu vẫn overfit (ví dụ: 1e-3)
    dropout_rate=0.3    # Tăng lên nếu vẫn overfit (tối đa 0.5)
)

# Trainer
trainer = pl.Trainer(
    max_epochs=500, 
    accelerator="auto", # Tự động dùng GPU nếu có
    callbacks=[checkpoint_callback, early_stop_callback],
    enable_progress_bar=True
)
# Bắt đầu huấn luyện
trainer.fit(model, train_loader, val_loader)
# Sau khi train, bạn có thể lấy embedding bằng:
# entity_embeddings = model.entity_emb.weight.detach().cpu().numpy()

### 1.5 Save trained 

In [ ]:
# 1. Extract Embeddings from Model (move to CPU and convert to numpy)
entity_embeddings = model.entity_emb.weight.detach().cpu().numpy()
relation_embeddings = model.relation_emb.weight.detach().cpu().numpy()

# 2. Package everything into a dictionary
saved_data = {
    'entity_embeddings': entity_embeddings,      # (Num_Entities, dim)
    'relation_embeddings': relation_embeddings,  # (Num_Relations, dim)
}
# 3. Save to a single file
with open(f'./pickle/{name}_transE_embeddings_{timestamp}.pkl', 'wb') as f:
    pickle.dump(saved_data, f)
print("Embeddings and mappings saved successfully!")